# Formal Extension Run Notebook

- Runtime version: 2026.04
- GPU: NVIDIA A100-SXM4-80GB
- This notebook is designed to run after the main frozen study has been completed
- Exploratory extension direction

In [1]:
# re-run on session restart (lightweight dependencies)
from google.colab import userdata
import os
import json
from huggingface_hub import login

GITHUB_PAT, HF_PAT = userdata.get("GH_PAT"), userdata.get("HF_TOKEN-cl-prm-project")

login(HF_PAT)

In [2]:
# git remote set-url origin repo
!git clone https://{GITHUB_PAT}@github.com/Zhiheng-SC/cl-prm-project

Cloning into 'cl-prm-project'...
remote: Enumerating objects: 1273, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 1273 (delta 10), reused 9 (delta 1), pack-reused 1161 (from 3)
Receiving objects: 100% (1273/1273), 1.43 MiB | 20.33 MiB/s, done.
Resolving deltas: 100% (520/520), done.


In [2]:
# re-run on session restart
%cd cl-prm-project
!ls

/content/cl-prm-project
configs      outputs			requirements-common.txt  src
data	     pyproject.toml		requirements-runpod.txt  tests
docs	     README.md			requirements.txt
experiments  requirements-analysis.txt	scripts


In [4]:
!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
!pip install flash-attn==2.8.3.post1
!pip install -r requirements-runpod.txt

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 160.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 129.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 19.5 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.6.0
    Uninstalling triton-3.6.0:
      Successfully uninstalled triton-3.6.0
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nvidia-nccl-cu12 2.27.5
    Uninstalling nvidia-nccl-cu12-2.27.5:
      Successfully uninstalled nvidia-nccl-cu12-2.27.5
  Attempting uninstall: torch
    Found existing installation: torch 2.10.0+cu128
    Uninstalling torch-2.10.0+cu128:
      Successfully uninstalled torch-2.10.0+cu128
  Attempting uninstall: tor

In [3]:
!bash scripts/bootstrap_runpod.sh

Checking the image-provided CUDA binary stack...
Python: 3.12.13
Torch: 2.8.0+cu128
Torch CUDA: 12.8
FlashAttention: 2.8.3.post1
GPU: NVIDIA A100-SXM4-80GB
Kernel: (1, 128, 4, 64) torch.bfloat16 cuda:0
Installing RunPod-safe project dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 101.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Obtaining file:///content/cl-prm-project
  Checking if build backend supports build_editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cl-prm (pyproject.toml) ... done
  Created wheel for cl-prm: filename=cl_prm-0.1.0-0.editable-py3-none-any.whl size=1204 sha256=8901873d4f1e2a11f433b9b444d4feb74d34db7472f01b937770899ffb663f16
  Stored in directory: /tmp/pip-ephem-wheel-cache-q3gy11lg/wheels/4d/6e/d5/e7b30470ebf1f41dc1fbe9a1ba00e4f3ea58b215925e01810e
Successfully built cl-prm
Rec

- reconstruct splits without leakage from saved HF artifacts (verified with hash from manifest records!)

In [4]:
!hf download cl-prm-team/cl-prm-artifacts --repo-type dataset --local-dir ./outputs

Fetching 68 files:   0% 0/68 [00:00<?, ?it/s]Still waiting to acquire lock on outputs/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)

calibration_bins.csv: 100% 972/972 [00:00<00:00, 5.84MB/s]

call_outcomes_20pct.csv: 100% 424/424 [00:00<00:00, 3.79MB/s]
Download complete. Moving file to outputs/evaluation/router/final_analysis/calibration_bins.csv
Download complete. Moving file to outputs/evaluation/router/final_analysis/call_outcomes_20pct.csv

.gitattributes: 2.50kB [00:00, 4.45MB/s]
Download complete. Moving file to outputs/.gitattributes
Fetching 68 files:   1% 1/68 [00:00<00:36,  1.83it/s]
calibration_summary.csv: 100% 158/158 [00:00<00:00, 2.06MB/s]
Download complete. Moving file to outputs/evaluation/router/final_analysis/calibration_summary.csv

accuracy_vs_runtime.pdf: 17.0kB [00:00, 3.27MB/s]
Download complete. Moving file to outputs/evaluation/router/final_analysis/figures/accuracy_vs_runtime.pdf

README.md: 5.37kB [00:00, 1.89MB/s]
Download complete. Moving fil

In [20]:
import json
from pathlib import Path
import hashlib


def recover_split(split: str, base_dir: str = "outputs/formal") -> Path:
    base_path = Path(base_dir)
    input_path = base_path / split / f"pathfinder.jsonl"
    output_path = Path("data") / "prm_router"/ "formal" / (split + ".jsonl")
    meta_path =  base_path / split / "pathfinder.metadata.json"

    with meta_path.open("r", encoding="utf-8") as f:
         metadata = json.load(f)
         expected_num_examples, expected_sha256 = metadata["input"]["selected_examples"], metadata["input"]["sha256"]

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with input_path.open("r", encoding="utf-8") as source, \
         output_path.open("w", encoding="utf-8", newline="\n") as target:

        actual_num_examples = 0
        for line in source:
            if not line.strip():
                continue

            actual_num_examples += 1
            record = json.loads(line)
            original_record = {
                key: value
                for key, value in record.items()
                if not key.startswith("pathfinder_")
            }
            target.write(json.dumps(original_record, ensure_ascii=True) + "\n")

    if actual_num_examples != expected_num_examples:
       raise ValueError("reconstruction not matching")

    actual_sha256 = hashlib.sha256(output_path.read_bytes()).hexdigest()

    if actual_sha256 != expected_sha256:
       raise ValueError("reconstruction not matching")

    print(f"{split} reconstruction matches byte-for-byte!")
    print(f"outputs saved to: {output_path.resolve()}")

    return output_path

train_path = recover_split("train")
validation_path = recover_split("validation")

train reconstruction matches byte-for-byte!
outputs saved to: /content/cl-prm-project/data/prm_router/formal/train.jsonl
validation reconstruction matches byte-for-byte!
outputs saved to: /content/cl-prm-project/data/prm_router/formal/validation.jsonl


### Math-PRM - Pilot Run

In [18]:
!python experiments/extensions/prm_router/run_extensions.py \
  --split validation --models math_prm --limit 5

/usr/bin/python3 /content/cl-prm-project/experiments/extensions/prm_router/run_alternative.py --config /content/cl-prm-project/configs/experiments/prm_router_extensions.json --verifier math_prm --input /content/cl-prm-project/data/prm_router/formal/validation.jsonl --output /content/cl-prm-project/outputs/prm_router/extensions/pilot/validation/math_prm.jsonl --resume --limit 5


In [19]:
!python experiments/extensions/prm_router/run_extensions.py \
  --split validation --models math_prm --limit 5 --execute

/usr/bin/python3 /content/cl-prm-project/experiments/extensions/prm_router/run_alternative.py --config /content/cl-prm-project/configs/experiments/prm_router_extensions.json --verifier math_prm --input /content/cl-prm-project/data/prm_router/formal/validation.jsonl --output /content/cl-prm-project/outputs/prm_router/extensions/pilot/validation/math_prm.jsonl --resume --limit 5
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
Fetching 17 files:   0% 0/17 [00:00<?, ?it/s]
configuration.json: 100% 48.0/48.0 [00:00<00:00, 20.9kB/s]

README.md: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]


LICENSE: 6.96kB [00:00, 2.40MB/s]
README.md: 6.06kB [00:00, 452kB/s]


configuration_qwen2_rm.py: 6.68kB [00:00, 1.74MB/s]

config.json:   0% 0.00/835 [00:00<?, ?B/s]


config.json: 100% 835/835 [00:0

### Skywork-o1-PRM - Pilot Run

In [21]:
!git clone https://github.com/SkyworkAI/skywork-o1-prm-inference.git ../skywork-o1-prm-inference
!git -C ../skywork-o1-prm-inference checkout --detach 719b56b17447405e0f10e6c0360a581cf4ffa9c1

Cloning into '../skywork-o1-prm-inference'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 29 (delta 7), reused 5 (delta 5), pack-reused 15 (from 1)
Receiving objects: 100% (29/29), 23.80 KiB | 23.80 MiB/s, done.
Resolving deltas: 100% (8/8), done.
HEAD is now at 719b56b [feat] update hf inference & vllm inference example


In [22]:
!python experiments/extensions/prm_router/run_extensions.py \
  --split validation --models skywork_prm --limit 5

/usr/bin/python3 /content/cl-prm-project/experiments/extensions/prm_router/run_alternative.py --config /content/cl-prm-project/configs/experiments/prm_router_extensions.json --verifier skywork_prm --input /content/cl-prm-project/data/prm_router/formal/validation.jsonl --output /content/cl-prm-project/outputs/prm_router/extensions/pilot/validation/skywork_prm.jsonl --resume --limit 5


In [23]:
!python experiments/extensions/prm_router/run_extensions.py \
  --split validation --models skywork_prm --limit 5 --execute

/usr/bin/python3 /content/cl-prm-project/experiments/extensions/prm_router/run_alternative.py --config /content/cl-prm-project/configs/experiments/prm_router_extensions.json --verifier skywork_prm --input /content/cl-prm-project/data/prm_router/formal/validation.jsonl --output /content/cl-prm-project/outputs/prm_router/extensions/pilot/validation/skywork_prm.jsonl --resume --limit 5
Fetching 21 files:   0% 0/21 [00:00<?, ?it/s]
README.md: 21.0kB [00:00, 46.3MB/s]

.gitattributes: 0.00B [00:00, ?B/s]

.gitattributes: 1.57kB [00:00, 444kB/s]
added_tokens.json: 100% 605/605 [00:00<00:00, 477kB/s]

Fetching 21 files:   5% 1/21 [00:00<00:05,  3.64it/s]

main_result_code.png: 100% 71.5k/71.5k [00:00<00:00, 13.0MB/s]
configuration_qwen2_rm.py: 6.68kB [00:00, 1.75MB/s]

merges.txt: 1.67MB [00:00, 101MB/s]

config.json: 100% 851/851 [00:00<00:00, 6.34MB/s]
Fetching 21 files:  19% 4/21 [00:00<00:01, 11.87it/s]
main_result_math.png: 100% 131k/131k [00:00<00:00, 33.9MB/s]

prm_result_code.png:   0

### Full-Run

- `train`
- `validation`

In [24]:
!python experiments/extensions/prm_router/run_extensions.py \
  --split train --models math_prm skywork_prm --execute

!python experiments/extensions/prm_router/run_extensions.py \
  --split validation --models math_prm skywork_prm --execute

/usr/bin/python3 /content/cl-prm-project/experiments/extensions/prm_router/run_alternative.py --config /content/cl-prm-project/configs/experiments/prm_router_extensions.json --verifier math_prm --input /content/cl-prm-project/data/prm_router/formal/train.jsonl --output /content/cl-prm-project/outputs/prm_router/extensions/full/train/math_prm.jsonl --resume
Fetching 17 files: 100% 17/17 [00:00<00:00, 25293.78it/s]
2026-09-25 09:30:12.195711: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-25 09:30:12.217610: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790328612.240393    6436 cuda_dnn.cc:8579] Unable to register cuDN

* reuse RE and RE threshold from HF artifacts

In [26]:
with open("outputs/evaluation/router/reasoneval_threshold.json", "r") as f:
    reasoneval_threshold = json.load(f)
    threshold = reasoneval_threshold["selected_threshold"]
    print(f"reusing selected threhold: {threshold}")

reusing selected threhold: 0.9469655402936041


In [27]:
!python experiments/extensions/prm_router/evaluate_development.py \
  --model math_prm \
  --train-reasoneval outputs/formal/train/reasoneval.jsonl \
  --validation-reasoneval outputs/formal/validation/reasoneval.jsonl \
  --output outputs/prm_router/extensions/full/evaluation/math_prm_development_selection.json \
  --reason-eval-threshold {threshold}

{
  "model": "math_prm",
  "validation": {
    "reasoneval_accuracy": 0.725,
    "second_stage_accuracy": 0.655,
    "oracle_accuracy": 0.895,
    "beneficial": 34,
    "neutral": 118,
    "harmful": 48
  },
  "selected": {
    "lambda_h": 1.5,
    "mu": 0.5,
    "validation_accuracy": 0.795,
    "measured_extra_runtime_seconds": 1.338340417999916,
    "called_examples": 40
  },
  "cost_predictor": {
    "model": "ridge_regression",
    "validation_mae_seconds": 0.002050686841579468,
    "validation_r2": 0.9078501346088332,
    "median_baseline_mae_seconds": 0.004801473560013392,
    "train_median_runtime_seconds": 0.032515281499968296
  }
}
Saved to outputs/prm_router/extensions/full/evaluation/math_prm_development_selection.json


In [28]:
!python experiments/extensions/prm_router/evaluate_development.py \
  --model skywork_prm \
  --train-reasoneval outputs/formal/train/reasoneval.jsonl \
  --validation-reasoneval outputs/formal/validation/reasoneval.jsonl \
  --output outputs/prm_router/extensions/full/evaluation/skywork_prm_development_selection.json \
  --reason-eval-threshold {threshold}

{
  "model": "skywork_prm",
  "validation": {
    "reasoneval_accuracy": 0.725,
    "second_stage_accuracy": 0.68,
    "oracle_accuracy": 0.91,
    "beneficial": 37,
    "neutral": 117,
    "harmful": 46
  },
  "selected": {
    "lambda_h": 2,
    "mu": 0.5,
    "validation_accuracy": 0.77,
    "measured_extra_runtime_seconds": 1.4320311189994754,
    "called_examples": 40
  },
  "cost_predictor": {
    "model": "ridge_regression",
    "validation_mae_seconds": 0.0022430667393448034,
    "validation_r2": 0.927936053099965,
    "median_baseline_mae_seconds": 0.00478103936999787,
    "train_median_runtime_seconds": 0.03349984449994281
  }
}
Saved to outputs/prm_router/extensions/full/evaluation/skywork_prm_development_selection.json


In [30]:
recover_split("test")

test reconstruction matches byte-for-byte!
outputs saved to: /content/cl-prm-project/data/prm_router/formal/test.jsonl


PosixPath('data/prm_router/formal/test.jsonl')

In [31]:
!python experiments/extensions/prm_router/run_extensions.py \
  --split test --models math_prm skywork_prm

/usr/bin/python3 /content/cl-prm-project/experiments/extensions/prm_router/run_alternative.py --config /content/cl-prm-project/configs/experiments/prm_router_extensions.json --verifier math_prm --input /content/cl-prm-project/data/prm_router/formal/test.jsonl --output /content/cl-prm-project/outputs/prm_router/extensions/full/test/math_prm.jsonl --resume
/usr/bin/python3 /content/cl-prm-project/experiments/extensions/prm_router/run_alternative.py --config /content/cl-prm-project/configs/experiments/prm_router_extensions.json --verifier skywork_prm --input /content/cl-prm-project/data/prm_router/formal/test.jsonl --output /content/cl-prm-project/outputs/prm_router/extensions/full/test/skywork_prm.jsonl --resume


In [32]:
!python experiments/extensions/prm_router/run_extensions.py \
  --split test --models math_prm skywork_prm --confirm-exploratory-test --execute

/usr/bin/python3 /content/cl-prm-project/experiments/extensions/prm_router/run_alternative.py --config /content/cl-prm-project/configs/experiments/prm_router_extensions.json --verifier math_prm --input /content/cl-prm-project/data/prm_router/formal/test.jsonl --output /content/cl-prm-project/outputs/prm_router/extensions/full/test/math_prm.jsonl --resume
Fetching 17 files: 100% 17/17 [00:00<00:00, 23233.36it/s]
2026-09-25 09:38:53.224477: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-25 09:38:53.245608: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790329133.268534    9203 cuda_dnn.cc:8579] Unable to register cuDNN 

In [33]:
!hf upload cl-prm-team/cl-prm-artifacts \
  outputs/prm_router/extensions \
  extensions \
  --repo-type dataset

Start hashing 18 files.
Finished hashing 18 files.
https://huggingface.co/datasets/cl-prm-team/cl-prm-artifacts/tree/main/extensions
